# 08. Planejamentos e margens de segurança

Compara artefato, reservas e quantitativo por ano e gera as margens e as fichas de decisão. O código está em `funcoes/validacao.py` e `funcoes/margens.py`.

In [1]:
from funcoes.margens import avaliar_margem_por_contexto, gerar_margens_e_fichas, guardar_previsoes_da_semana
from funcoes.validacao import comparar_planejamentos

comparar_planejamentos()
gerar_margens_e_fichas()

 ano    alternativa  dias   mae   vies  excesso  deficit  dias_deficit  dif_plano  ic_inf  ic_sup
2023          plano   166 16.11   2.10   1511.0   1163.0         40.36        NaN     NaN     NaN
2023 consumo_tipico   166 16.41   6.52   1903.0    821.0         47.59       0.30   -3.64    5.29
2024          plano   118 39.89 -34.23    334.0   4373.0         82.20        NaN     NaN     NaN
2024 consumo_tipico   118 21.48  -0.33   1248.0   1287.0         50.85     -18.41  -26.21  -10.36
2024       artefato   118 24.10   4.78   1704.0   1140.0         44.92     -15.79  -23.88   -7.69
2024     combinacao   118 23.44  -5.76   1043.0   1723.0         58.47     -16.45  -23.55   -9.38
2025          plano   152 28.85 -21.10    589.0   3796.0         80.26        NaN     NaN     NaN
2025 consumo_tipico   152 20.64   4.11   1881.0   1257.0         46.05      -8.20  -13.93   -1.19
2025       artefato   152 20.02   2.59   1718.0   1325.0         46.71      -8.83  -13.66   -3.38
2025     combinacao 

decisao      universo         opcao  dias  erro_medio  folga_media  a_mais  a_menos  dias_a_menos_pct  prometido_dias_a_menos_pct
preparo  sem_atipicos     sem_folga   121        24.2          0.0  1501.0   1430.0              52.1                        51.4
preparo  sem_atipicos      moderada   121        26.9         14.7  2561.0    688.0              28.9                        24.3
preparo  sem_atipicos          alta   121        37.9         31.9  4273.0    317.0              13.2                         9.7
preparo  sem_atipicos nutricionista   121        24.1          NaN  2255.0    663.0              39.7                         NaN
preparo todos_os_dias     sem_folga   123        26.5          0.0  1834.0   1430.0              51.2                        51.4
preparo todos_os_dias      moderada   123        29.4         14.8  2927.0    688.0              28.5                        24.3
preparo todos_os_dias          alta   123        40.6         31.9  4671.0    317.0       

In [2]:
import json
import sys
from pathlib import Path

import pandas as pd
from IPython.display import Markdown, display

from funcoes.nomes import FOLGAS, UNIVERSOS

pd.set_option("display.max_rows", 60, "display.max_columns", 20, "display.width", 180)
SAIDA = Path("..") / "resultados" / "planejamento"
v = pd.read_csv(SAIDA / "verificacao_folgas_2026.csv", sep=";", encoding="utf-8-sig")
s = pd.read_csv(SAIDA / "verificacao_por_situacao_2026.csv", sep=";", encoding="utf-8-sig")
NOMES_OPCAO = {**FOLGAS, "nutricionista": "Quantitativo da nutricionista"}

## 1. As margens de segurança protegem como prometem?

A coluna **prometido** é a estimativa feita antes de cada semana; **observado** é o que aconteceu de fevereiro a agosto de 2026, sem dias atípicos.

In [3]:
d = v[v["universo"] == "sem_atipicos"].copy()
d["Opção"] = d["opcao"].map(NOMES_OPCAO)
d = d.rename(columns={"decisao": "Decisão", "folga_media": "Margem média", "prometido_dias_a_menos_pct": "Déficit potencial, prometido (% dos dias)",
                      "dias_a_menos_pct": "Déficit potencial, observado (% dos dias)", "a_mais": "Excesso potencial (refeições)", "a_menos": "Déficit potencial (refeições)"})
d[["Decisão", "Opção", "Margem média", "Déficit potencial, prometido (% dos dias)", "Déficit potencial, observado (% dos dias)", "Excesso potencial (refeições)", "Déficit potencial (refeições)"]].round(1)

,Decisão,Opção,Margem média,"Déficit potencial, prometido (% dos dias)","Déficit potencial, observado (% dos dias)",Excesso potencial (refeições),Déficit potencial (refeições)
0,preparo,Sem margem,0.0,51.4,52.1,1501.0,1430.0
1,preparo,Margem moderada,14.7,24.3,28.9,2561.0,688.0
2,preparo,Margem alta,31.9,9.7,13.2,4273.0,317.0
3,preparo,Quantitativo da nutricionista,NaN,NaN,39.7,2255.0,663.0
8,compras,Sem margem,0.0,54.7,57.0,1363.0,1485.0
9,compras,Margem moderada,14.1,24.5,28.1,2355.0,740.0
10,compras,Margem alta,32.0,9.6,13.2,4113.0,338.0


## 2. Por que a margem de segurança precisa de contexto

A mesma margem de segurança não protege igual em todo dia. Em dia atípico, o consumo fica muito abaixo do previsto e qualquer margem de segurança só aumenta a diferença. `Tendência da previsão` positiva indica previsão acima do consumo.

In [4]:
t = s.rename(columns={"decisao": "Decisão", "situacao": "Situação do dia", "dias": "Dias", "erro_previsao": "Tendência da previsão (refeições)",
                      "dias_a_menos_sem_folga": "Sem margem", "dias_a_menos_moderada": "Margem moderada", "dias_a_menos_alta": "Margem alta"})
t[["Decisão", "Situação do dia", "Dias", "Tendência da previsão (refeições)", "Sem margem", "Margem moderada", "Margem alta"]].round(1)

,Decisão,Situação do dia,Dias,Tendência da previsão (refeições),Sem margem,Margem moderada,Margem alta
0,preparo,Dia atípico,2,166.5,0.0,0.0,0.0
1,preparo,Dia comum,102,-3.6,55.9,30.4,13.7
2,preparo,Início do período ou volta de pausa,15,22.7,33.3,26.7,13.3
3,preparo,Últimos dias do período,4,23.5,25.0,0.0,0.0
4,compras,Dia atípico,2,165.5,0.0,0.0,0.0
5,compras,Dia comum,102,-5.7,61.8,30.4,13.7
6,compras,Início do período ou volta de pausa,15,24.6,33.3,20.0,13.3
7,compras,Últimos dias do período,4,23.8,25.0,0.0,0.0


## 3. Fichas da próxima emissão

Exemplo do que a nutricionista e a gestora receberiam na sexta-feira mais recente.

In [5]:
arquivo = sorted(SAIDA.glob("fichas_*.md"))[-1]
display(Markdown(arquivo.read_text(encoding="utf-8")))

# Planejamento de refeições: sexta-feira, 30/10/2026

As opções e os dados abaixo apoiam a decisão. Quem define a quantidade é a nutricionista, com a gestora.

## Preparo (semana seguinte)

### Terça-feira, 03/11/2026

**Previsão: 194 refeições** (Cenário 1: consumo e calendário)

| Opção | Refeições | Margem de segurança | Chance de déficit potencial |
|---|---:|---:|---:|
| Sem margem | 194 | +0 | 43% |
| Margem moderada | 205 | +11 | 19% |
| Margem alta | 222 | +28 | 10% |

- **Quantitativo da nutricionista:** 210. Combinado com a previsão: 197.
- **Calendário:** Aula regular; 4 dias de aula na semana.
- **Cardápio:** ainda não informado.
- **Clima:** Chuva prevista: 0 mm no dia, 0 mm de manhã; máxima de 29 °C.
- **Histórico:** Consumo de referência de 194; mesmo dia nas últimas vezes: 192, 185, 199, 201. Na mesma época do ano passado, o consumo ficou em 98% da referência daquele momento.
- **Dias parecidos:** Volta depois de dia sem aula: consumo em torno de 91% da referência (32 dias anteriores).
- **Atenção:** Dias parecidos costumam ficar abaixo da referência: confira antes de usar margem de segurança.

### Quarta-feira, 04/11/2026

**Previsão: 225 refeições** (Cenário 1: consumo e calendário)

| Opção | Refeições | Margem de segurança | Chance de déficit potencial |
|---|---:|---:|---:|
| Sem margem | 225 | +0 | 50% |
| Margem moderada | 242 | +17 | 22% |
| Margem alta | 256 | +31 | 10% |

- **Quantitativo da nutricionista:** 230. Combinado com a previsão: 226.
- **Calendário:** Aula regular; 4 dias de aula na semana.
- **Cardápio:** ainda não informado.
- **Clima:** Chuva prevista: 5 mm no dia, 1 mm de manhã; máxima de 29 °C.
- **Histórico:** Consumo de referência de 200; mesmo dia nas últimas vezes: 232, 202, 193, 215. Na mesma época do ano passado, o consumo ficou em 97% da referência daquele momento.

### Quinta-feira, 05/11/2026

**Previsão: 190 refeições** (Cenário 1: consumo e calendário)

| Opção | Refeições | Margem de segurança | Chance de déficit potencial |
|---|---:|---:|---:|
| Sem margem | 190 | +0 | 50% |
| Margem moderada | 207 | +17 | 22% |
| Margem alta | 221 | +31 | 10% |

- **Quantitativo da nutricionista:** 200. Combinado com a previsão: 192.
- **Calendário:** Aula regular; 4 dias de aula na semana.
- **Cardápio:** ainda não informado.
- **Clima:** Chuva prevista: 7 mm no dia, 2 mm de manhã; máxima de 26 °C.
- **Histórico:** Consumo de referência de 195; mesmo dia nas últimas vezes: 172, 193, 202, 210. Na mesma época do ano passado, o consumo ficou em 97% da referência daquele momento.
- **Dias parecidos:** Quintas-feiras: consumo em torno de 95% da referência (127 dias anteriores).

### Sexta-feira, 06/11/2026

**Previsão: 183 refeições** (Cenário 1: consumo e calendário)

| Opção | Refeições | Margem de segurança | Chance de déficit potencial |
|---|---:|---:|---:|
| Sem margem | 183 | +0 | 50% |
| Margem moderada | 200 | +17 | 22% |
| Margem alta | 214 | +31 | 10% |

- **Quantitativo da nutricionista:** 190. Combinado com a previsão: 184.
- **Calendário:** Aula regular; 4 dias de aula na semana.
- **Cardápio:** ainda não informado.
- **Clima:** Chuva prevista: 6 mm no dia, 0 mm de manhã; máxima de 26 °C.
- **Histórico:** Consumo de referência de 190; mesmo dia nas últimas vezes: 149, 181, 194, 201. Na mesma época do ano passado, o consumo ficou em 97% da referência daquele momento.

## Compras (daqui a duas semanas)

### Segunda-feira, 09/11/2026

**Previsão: 229 refeições** (Cenário 1: consumo e calendário)

| Opção | Refeições | Margem de segurança | Chance de déficit potencial |
|---|---:|---:|---:|
| Sem margem | 229 | +0 | 56% |
| Margem moderada | 244 | +15 | 25% |
| Margem alta | 263 | +34 | 10% |

- **Calendário:** Aula regular; 5 dias de aula na semana.
- **Cardápio:** ainda não informado.
- **Clima:** Chuva prevista: 2 mm no dia, 1 mm de manhã; máxima de 22 °C. Com duas semanas de antecedência, essa previsão é pouco confiável.
- **Histórico:** Consumo de referência de 202; mesmo dia nas últimas vezes: 199, 224, 201, 237. Na mesma época do ano passado, o consumo ficou em 95% da referência daquele momento.

### Terça-feira, 10/11/2026

**Previsão: 207 refeições** (Cenário 1: consumo e calendário)

| Opção | Refeições | Margem de segurança | Chance de déficit potencial |
|---|---:|---:|---:|
| Sem margem | 207 | +0 | 56% |
| Margem moderada | 222 | +15 | 25% |
| Margem alta | 241 | +34 | 10% |

- **Calendário:** Aula regular; 5 dias de aula na semana.
- **Cardápio:** ainda não informado.
- **Clima:** Chuva prevista: 1 mm no dia, 0 mm de manhã; máxima de 22 °C. Com duas semanas de antecedência, essa previsão é pouco confiável.
- **Histórico:** Consumo de referência de 194; mesmo dia nas últimas vezes: 192, 185, 199, 201. Na mesma época do ano passado, o consumo ficou em 93% da referência daquele momento.

### Quarta-feira, 11/11/2026

**Previsão: 225 refeições** (Cenário 1: consumo e calendário)

| Opção | Refeições | Margem de segurança | Chance de déficit potencial |
|---|---:|---:|---:|
| Sem margem | 225 | +0 | 56% |
| Margem moderada | 240 | +15 | 25% |
| Margem alta | 259 | +34 | 10% |

- **Calendário:** Aula regular; 5 dias de aula na semana.
- **Cardápio:** ainda não informado.
- **Clima:** Chuva prevista: 21 mm no dia, 1 mm de manhã; máxima de 23 °C. Com duas semanas de antecedência, essa previsão é pouco confiável.
- **Histórico:** Consumo de referência de 200; mesmo dia nas últimas vezes: 232, 202, 193, 215. Na mesma época do ano passado, o consumo ficou em 94% da referência daquele momento.

### Quinta-feira, 12/11/2026

**Previsão: 201 refeições** (Cenário 1: consumo e calendário)

| Opção | Refeições | Margem de segurança | Chance de déficit potencial |
|---|---:|---:|---:|
| Sem margem | 201 | +0 | 56% |
| Margem moderada | 216 | +15 | 25% |
| Margem alta | 235 | +34 | 10% |

- **Calendário:** Aula regular; 5 dias de aula na semana.
- **Cardápio:** ainda não informado.
- **Clima:** Chuva prevista: 9 mm no dia, 1 mm de manhã; máxima de 24 °C. Com duas semanas de antecedência, essa previsão é pouco confiável.
- **Histórico:** Consumo de referência de 195; mesmo dia nas últimas vezes: 172, 193, 202, 210. Na mesma época do ano passado, o consumo ficou em 93% da referência daquele momento.
- **Dias parecidos:** Quintas-feiras: consumo em torno de 95% da referência (127 dias anteriores).

### Sexta-feira, 13/11/2026

**Previsão: 195 refeições** (Cenário 1: consumo e calendário)

| Opção | Refeições | Margem de segurança | Chance de déficit potencial |
|---|---:|---:|---:|
| Sem margem | 195 | +0 | 56% |
| Margem moderada | 210 | +15 | 25% |
| Margem alta | 229 | +34 | 10% |

- **Calendário:** Aula regular; 5 dias de aula na semana.
- **Cardápio:** ainda não informado.
- **Clima:** Chuva prevista: 6 mm no dia, 1 mm de manhã; máxima de 26 °C. Com duas semanas de antecedência, essa previsão é pouco confiável.
- **Histórico:** Consumo de referência de 190; mesmo dia nas últimas vezes: 149, 181, 194, 201. Na mesma época do ano passado, o consumo ficou em 93% da referência daquele momento.


## Margem de segurança por contexto

Usa os erros de dias parecidos nos últimos 12 meses. Com menos de 15 dias parecidos, usa a margem geral. A comparação abaixo levou à adoção desta margem: ela reduziu o erro em 2025 e não piorou 2026.

In [6]:
em_2025, ate_agosto, prospectiva, por_situacao = avaliar_margem_por_contexto()
display(em_2025, ate_agosto, prospectiva, por_situacao)

,opcao,dias,excesso,deficit,dias_com_deficit_pct,erro_medio
0,"Margem geral, moderada",145,3355,384,21.4,25.8
1,"Margem geral, alta",145,5755,94,6.2,40.3
2,"Margem por contexto, moderada",145,3059,415,22.8,24.0
3,"Margem por contexto, alta",145,5475,88,6.9,38.4


,opcao,dias,excesso,deficit,dias_com_deficit_pct,erro_medio
0,"Margem geral, moderada",121,2841,666,24.8,29.0
1,"Margem geral, alta",121,4707,245,10.7,40.9
2,"Margem por contexto, moderada",121,2606,657,28.9,27.0
3,"Margem por contexto, alta",121,4476,249,11.6,39.0


,opcao,dias,excesso,deficit,dias_com_deficit_pct,erro_medio
0,"Margem geral, moderada",28,406,44,28.6,16.1
1,"Margem geral, alta",28,654,7,3.6,23.6
2,"Margem por contexto, moderada",28,494,52,25.0,19.5
3,"Margem por contexto, alta",28,936,0,0.0,33.4


,dias,deficit_geral_pct,deficit_contexto_pct
situacao,,,
comum,97.0,26.8,30.9
fim,4.0,0.0,0.0
inicio,15.0,20.0,26.7
vespera,5.0,20.0,20.0


## Guardar as previsões da semana

Função usada na operação semanal. O estudo vai até outubro de 2026. Previsões já guardadas nunca são substituídas.

In [7]:
guardar_previsoes_da_semana()

Previsoes de 2026-10-30 ja guardadas: nada foi alterado
